# XHPDF-style Reproduction Notebook

This notebook implements an **XHPDF-style discourse-aware AES pipeline** based on the paper setup:

- Prompts 1–7: compressed / sentence embedding model proxy
- Prompt 8: XLNet embedding model
- Extracted discourse-style features
- Concatenate `[embedding ; scaled discourse features]`
- Linear regression head
- Evaluation with QWK using both `round` and validation-optimized thresholds

Important: unless you have the exact HPD checkpoint/code from the paper, this is best reported as **XHPDF-style reproduced / approximate**, not exact original XHPDF.


In [1]:
# ============================================================
# 0. SET PATHS FOR COLAB RUNTIME
# ============================================================
# Upload these 3 CSV files to Colab /content first:
#   - asap_train.csv
#   - asap_val.csv
#   - asap_test.csv
#
# If your filenames are different, edit the paths below.

from pathlib import Path

TRAIN_PATH = Path('/content/asap_train.csv')
VAL_PATH   = Path('/content/asap_val.csv')
TEST_PATH  = Path('/content/asap_test.csv')

for p in [TRAIN_PATH, VAL_PATH, TEST_PATH]:
    if not p.exists():
        raise FileNotFoundError(
            f'Missing file: {p}\n'
            'Please upload the 3 CSV files to Colab /content, or edit TRAIN_PATH/VAL_PATH/TEST_PATH.'
        )

print('Found files:')
print('TRAIN:', TRAIN_PATH)
print('VAL  :', VAL_PATH)
print('TEST :', TEST_PATH)


Found files:
TRAIN: /content/asap_train.csv
VAL  : /content/asap_val.csv
TEST : /content/asap_test.csv


In [2]:
# ============================================================
# 1. INSTALL / IMPORT DEPENDENCIES
# ============================================================
!pip -q install transformers accelerate scikit-learn nltk tqdm

import os
import re
import math
import random
import warnings
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from transformers import AutoTokenizer, AutoModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


Device: cuda


In [3]:
# ============================================================
# 2. LOAD TRAIN / VAL / TEST CSV FILES
# ============================================================
train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print('train_df:', train_df.shape)
print('val_df  :', val_df.shape)
print('test_df :', test_df.shape)
display(train_df.head())

# Required columns expected by this notebook.
# If your files use different names, edit these variables.
ESSAY_SET_COL = 'essay_set'
TEXT_COL = 'essay'
SCORE_COL = 'domain1_score'

required_cols = {ESSAY_SET_COL, TEXT_COL, SCORE_COL}
for name, df in [('train_df', train_df), ('val_df', val_df), ('test_df', test_df)]:
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f'{name} is missing columns: {missing}. Current columns: {list(df.columns)}')

# Clean essential columns.
for df in [train_df, val_df, test_df]:
    df[ESSAY_SET_COL] = df[ESSAY_SET_COL].astype(int)
    df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)
    df[SCORE_COL] = df[SCORE_COL].astype(float)

print('Essay sets:', sorted(train_df[ESSAY_SET_COL].unique()))


train_df: (9084, 5)
val_df  : (1296, 5)
test_df : (2596, 5)


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,14876,6,In the excerpt The Mooring Mast by Marcia Amid...,3.0,0.75
1,9985,4,The author concluded this sentence because he ...,0.0,0.00
2,3231,2,"I can think of several books that, I would not...",4.0,0.60
3,21137,8,My best friend @PERSON2 turned thirteen on a b...,39.0,0.65
4,17919,7,A time that I was patient is when I broke my f...,23.0,0.77


Essay sets: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]


In [4]:
# ============================================================
# 3. CONFIG: XHPDF-STYLE MODELS
# ============================================================
# Paper idea:
# - prompts 1-7: HPD / compressed sentence embedding model
# - prompt 8: XLNet
#
# Exact HPD checkpoint from the paper may not be public or may have a different name.
# So this notebook exposes HPD_MODEL_NAME as an editable variable.
# Default is a lightweight sentence-transformer checkpoint used as a practical proxy.
#
# If you find the exact HPD checkpoint/code, replace HPD_MODEL_NAME below.

HPD_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'  # proxy for compressed sentence embeddings
XLNET_MODEL_NAME = 'xlnet-base-cased'

# Maximum token length per prompt.
# The paper changes max sequence length per prompt; these are practical Colab-safe defaults.
MAX_LENGTH_BY_PROMPT = {
    1: 512,
    2: 512,
    3: 256,
    4: 256,
    5: 256,
    6: 256,
    7: 512,
    8: 512,
}

BATCH_SIZE_EMBED = 8
BATCH_SIZE_HEAD = 64
HEAD_EPOCHS = 80
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 12

OUTPUT_DIR = Path('/content/benchmark_results/ASAP')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('HPD_MODEL_NAME  :', HPD_MODEL_NAME)
print('XLNET_MODEL_NAME:', XLNET_MODEL_NAME)
print('OUTPUT_DIR      :', OUTPUT_DIR)


HPD_MODEL_NAME  : sentence-transformers/all-MiniLM-L6-v2
XLNET_MODEL_NAME: xlnet-base-cased
OUTPUT_DIR      : /content/benchmark_results/ASAP


In [5]:
# ============================================================
# 4. QWK HELPERS: ROUND AND OPTIMIZED THRESHOLDS
# ============================================================
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights='quadratic')

def clip_and_round(preds, min_score, max_score):
    preds = np.clip(preds, min_score, max_score)
    return np.rint(preds).astype(int)

def apply_thresholds(preds, thresholds, min_score, max_score):
    thresholds = np.asarray(thresholds)
    classes = np.digitize(preds, thresholds) + min_score
    return np.clip(classes, min_score, max_score).astype(int)

def optimize_thresholds(y_true, raw_preds, min_score, max_score):
    # Need one threshold between each adjacent integer score.
    n_thresholds = int(max_score - min_score)
    if n_thresholds <= 0:
        return np.array([])

    init_thresholds = np.arange(min_score + 0.5, max_score + 0.5, 1.0)

    def objective(thresholds):
        thresholds = np.sort(thresholds)
        pred_classes = apply_thresholds(raw_preds, thresholds, min_score, max_score)
        return -qwk(y_true, pred_classes)

    result = minimize(objective, init_thresholds, method='Nelder-Mead',
                      options={'maxiter': 1000, 'disp': False})
    return np.sort(result.x)

def evaluate_predictions(y_true, raw_preds, min_score, max_score, thresholds=None):
    round_preds = clip_and_round(raw_preds, min_score, max_score)
    out = {'qwk_round': qwk(y_true, round_preds)}
    if thresholds is not None:
        opt_preds = apply_thresholds(raw_preds, thresholds, min_score, max_score)
        out['qwk_opt'] = qwk(y_true, opt_preds)
    return out


In [6]:
# ============================================================
# 5. EXTRACT DISCOURSE-STYLE FEATURES
# ============================================================
CONNECTIVES = {
    'addition': ['also', 'moreover', 'furthermore', 'besides', 'in addition', 'additionally'],
    'contrast': ['however', 'but', 'although', 'though', 'nevertheless', 'on the other hand', 'whereas'],
    'cause': ['because', 'since', 'therefore', 'thus', 'hence', 'consequently', 'as a result'],
    'example': ['for example', 'for instance', 'such as', 'including'],
    'sequence': ['first', 'second', 'third', 'finally', 'then', 'next', 'after', 'before'],
    'conclusion': ['in conclusion', 'to conclude', 'overall', 'in summary', 'to sum up']
}

STOPWORDS = set('''
a an the and or but if while with without of for to in on at by from up down out over under again further then once here there
all any both each few more most other some such no nor not only own same so than too very can will just should now is am are was were be been being
have has had do does did this that these those i you he she it we they me him her us them my your his its our their
'''.split())

def simple_sent_tokenize(text):
    sents = re.split(r'[.!?]+', str(text))
    return [s.strip() for s in sents if s.strip()]

def simple_word_tokenize(text):
    return re.findall(r"\b[a-zA-Z]+(?:'[a-zA-Z]+)?\b", str(text).lower())

def count_phrase(text_lower, phrase):
    return len(re.findall(r'\b' + re.escape(phrase) + r'\b', text_lower))

def safe_div(a, b):
    return float(a) / float(b) if b else 0.0

def flesch_reading_ease_proxy(words, sentences):
    # Lightweight readability proxy without syllable dictionary.
    # Counts vowel groups as approximate syllables.
    def syllables(word):
        groups = re.findall(r'[aeiouy]+', word.lower())
        return max(1, len(groups))
    n_words = len(words)
    n_sents = max(1, len(sentences))
    n_syll = sum(syllables(w) for w in words)
    return 206.835 - 1.015 * safe_div(n_words, n_sents) - 84.6 * safe_div(n_syll, n_words)

def lexical_chain_proxy_features(words, sentences):
    # Approximate lexical-chain style cohesion:
    # repeated content words across essay and sentence-span coverage.
    content_words = [w for w in words if w not in STOPWORDS and len(w) > 2]
    counts = Counter(content_words)
    repeated = {w: c for w, c in counts.items() if c >= 2}

    # Map word -> sentence indices where it appears.
    sent_words = [set(simple_word_tokenize(s)) for s in sentences]
    spans = []
    for w in repeated:
        idxs = [i for i, sw in enumerate(sent_words) if w in sw]
        if idxs:
            spans.append(max(idxs) - min(idxs) + 1)

    large_chains = [w for w, c in counts.items() if c >= 4]
    varied_chains = len(repeated)
    avg_chain_size = np.mean(list(repeated.values())) if repeated else 0.0
    avg_span = np.mean(spans) if spans else 0.0

    return {
        'lc_avg_chain_size_proxy': avg_chain_size,
        'lc_num_varied_chains_proxy': varied_chains,
        'lc_num_large_chains_proxy': len(large_chains),
        'lc_pct_large_chains_proxy': safe_div(len(large_chains), max(1, len(repeated))),
        'lc_avg_span_proxy': avg_span,
    }

def sentence_overlap_features(sentences):
    sent_sets = []
    for s in sentences:
        words = [w for w in simple_word_tokenize(s) if w not in STOPWORDS and len(w) > 2]
        sent_sets.append(set(words))

    overlaps = []
    jaccards = []
    for a, b in zip(sent_sets[:-1], sent_sets[1:]):
        if not a or not b:
            overlaps.append(0)
            jaccards.append(0.0)
        else:
            inter = len(a & b)
            union = len(a | b)
            overlaps.append(inter)
            jaccards.append(safe_div(inter, union))

    return {
        'adjacent_overlap_mean': np.mean(overlaps) if overlaps else 0.0,
        'adjacent_overlap_max': np.max(overlaps) if overlaps else 0.0,
        'adjacent_jaccard_mean': np.mean(jaccards) if jaccards else 0.0,
    }

def extract_discourse_features_one(text):
    text = str(text)
    text_lower = text.lower()
    sentences = simple_sent_tokenize(text)
    words = simple_word_tokenize(text)

    n_words = len(words)
    n_sentences = len(sentences)
    unique_words = set(words)
    content_words = [w for w in words if w not in STOPWORDS and len(w) > 2]

    feats = {}
    # Descriptive / length features
    feats['char_count'] = len(text)
    feats['word_count'] = n_words
    feats['sentence_count'] = n_sentences
    feats['avg_sentence_length'] = safe_div(n_words, n_sentences)
    feats['unique_word_count'] = len(unique_words)
    feats['type_token_ratio'] = safe_div(len(unique_words), n_words)
    feats['content_word_ratio'] = safe_div(len(content_words), n_words)

    # Punctuation / style
    feats['comma_count'] = text.count(',')
    feats['semicolon_count'] = text.count(';')
    feats['colon_count'] = text.count(':')
    feats['question_count'] = text.count('?')
    feats['exclamation_count'] = text.count('!')
    feats['quote_count'] = text.count('"') + text.count("'")
    feats['punctuation_ratio'] = safe_div(sum(1 for ch in text if ch in ',;:.!?'), max(1, len(text)))

    # Connective counts
    total_conn = 0
    for group, items in CONNECTIVES.items():
        c = sum(count_phrase(text_lower, item) for item in items)
        feats[f'connective_{group}_count'] = c
        total_conn += c
    feats['connective_total_count'] = total_conn
    feats['connective_per_100_words'] = 100.0 * safe_div(total_conn, n_words)

    # Readability proxy
    feats['flesch_reading_ease_proxy'] = flesch_reading_ease_proxy(words, sentences)

    # Cohesion proxies
    feats.update(sentence_overlap_features(sentences))
    feats.update(lexical_chain_proxy_features(words, sentences))

    return feats

def extract_discourse_features_df(df, text_col=TEXT_COL):
    rows = [extract_discourse_features_one(x) for x in tqdm(df[text_col].tolist(), desc='Extracting discourse features')]
    return pd.DataFrame(rows).fillna(0.0)

train_feat_df = extract_discourse_features_df(train_df)
val_feat_df   = extract_discourse_features_df(val_df)
test_feat_df  = extract_discourse_features_df(test_df)

print(train_feat_df.shape)
display(train_feat_df.head())


Extracting discourse features:   0%|          | 0/9084 [00:00<?, ?it/s]

Extracting discourse features:   0%|          | 0/1296 [00:00<?, ?it/s]

Extracting discourse features:   0%|          | 0/2596 [00:00<?, ?it/s]

(9084, 31)


,char_count,word_count,sentence_count,avg_sentence_length,unique_word_count,type_token_ratio,content_word_ratio,comma_count,semicolon_count,colon_count,...,connective_per_100_words,flesch_reading_ease_proxy,adjacent_overlap_mean,adjacent_overlap_max,adjacent_jaccard_mean,lc_avg_chain_size_proxy,lc_num_varied_chains_proxy,lc_num_large_chains_proxy,lc_pct_large_chains_proxy,lc_avg_span_proxy
0,1194,193,12,16.083333,119,0.616580,0.616580,8,0,0,...,1.036269,41.035805,0.545455,3.0,0.032290,2.600000,20,3,0.150000,6.500000
1,179,32,1,32.000000,28,0.875000,0.500000,1,0,0,...,3.125000,39.523750,0.000000,0.0,0.000000,0.000000,0,0,0.000000,0.000000
2,3915,739,53,13.943396,281,0.380244,0.473613,19,0,0,...,2.976996,80.264050,0.288462,2.0,0.022744,3.093750,64,14,0.218750,23.656250
3,4184,724,50,14.480000,310,0.428177,0.483425,21,0,0,...,2.348066,68.392772,0.122449,1.0,0.011655,2.894737,57,11,0.192982,21.017544
4,1180,255,19,13.421053,116,0.454902,0.439216,0,0,0,...,1.176471,86.052632,0.555556,2.0,0.053108,2.750000,20,4,0.200000,9.250000


In [7]:
# ============================================================
# 6. EMBEDDING EXTRACTION
# ============================================================
def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-9)
    return summed / denom

@torch.no_grad()
def encode_texts(texts, model_name, max_length=512, batch_size=8, pooling='cls'):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE)
    model.eval()

    all_embeddings = []
    for start in tqdm(range(0, len(texts), batch_size), desc=f'Encoding {model_name}'):
        batch_texts = texts[start:start + batch_size]
        enc = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors='pt'
        )
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        hidden = out.last_hidden_state

        if pooling == 'mean':
            emb = mean_pool(hidden, enc['attention_mask'])
        else:
            # CLS / first token representation, matching the paper's description.
            emb = hidden[:, 0, :]

        all_embeddings.append(emb.detach().cpu().numpy())

    del model
    torch.cuda.empty_cache()
    return np.vstack(all_embeddings)

def get_model_for_prompt(prompt_id):
    return HPD_MODEL_NAME, 'cls'

def encode_split_for_prompt(df_prompt, prompt_id):
    model_name, pooling = get_model_for_prompt(prompt_id)
    max_len = MAX_LENGTH_BY_PROMPT.get(int(prompt_id), 512)
    texts = df_prompt[TEXT_COL].tolist()
    return encode_texts(texts, model_name=model_name, max_length=max_len,
                        batch_size=BATCH_SIZE_EMBED, pooling=pooling)


In [8]:
# ============================================================
# 7. LINEAR HEAD MODEL
# ============================================================
class LinearRegressionHead(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        # Paper uses ReLU to map to non-negative score.
        return torch.relu(self.linear(x)).squeeze(-1)

def train_linear_head(X_train, y_train_norm, X_val, y_val_norm,
                      epochs=HEAD_EPOCHS, lr=HEAD_LR, weight_decay=WEIGHT_DECAY):
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_norm, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
    y_val_t = torch.tensor(y_val_norm, dtype=torch.float32).to(DEVICE)

    loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE_HEAD, shuffle=True)

    model = LinearRegressionHead(X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_val = float('inf')
    patience_counter = 0

    history = []
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_pred = model(X_val_t)
            val_loss = loss_fn(val_pred, y_val_t).item()

        history.append({'epoch': epoch, 'train_loss': float(np.mean(train_losses)), 'val_loss': val_loss})

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch % 10 == 0 or epoch == 1:
            print(f'Epoch {epoch:03d} | train_loss={np.mean(train_losses):.5f} | val_loss={val_loss:.5f}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

@torch.no_grad()
def predict_head(model, X):
    model.eval()
    X_t = torch.tensor(X, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t), batch_size=256, shuffle=False)
    preds = []
    for (xb,) in loader:
        xb = xb.to(DEVICE)
        preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)


In [9]:
# ============================================================
# 8. TRAIN / EVALUATE XHPDF-STYLE MODEL PER PROMPT
# ============================================================
all_results = []
all_predictions = []
all_histories = []

for prompt_id in sorted(train_df[ESSAY_SET_COL].unique()):
    print('\n' + '=' * 80)
    print(f'PROMPT {prompt_id}')
    print('=' * 80)

    tr_idx = train_df[train_df[ESSAY_SET_COL] == prompt_id].index
    va_idx = val_df[val_df[ESSAY_SET_COL] == prompt_id].index
    te_idx = test_df[test_df[ESSAY_SET_COL] == prompt_id].index

    tr_df = train_df.loc[tr_idx].reset_index(drop=True)
    va_df = val_df.loc[va_idx].reset_index(drop=True)
    te_df = test_df.loc[te_idx].reset_index(drop=True)

    tr_feat = train_feat_df.loc[tr_idx].reset_index(drop=True)
    va_feat = val_feat_df.loc[va_idx].reset_index(drop=True)
    te_feat = test_feat_df.loc[te_idx].reset_index(drop=True)

    min_score = int(min(tr_df[SCORE_COL].min(), va_df[SCORE_COL].min(), te_df[SCORE_COL].min()))
    max_score = int(max(tr_df[SCORE_COL].max(), va_df[SCORE_COL].max(), te_df[SCORE_COL].max()))
    score_range = max_score - min_score

    print(f'Score range: {min_score} to {max_score}')
    print('Rows:', len(tr_df), len(va_df), len(te_df))

    # 1) Encode essays using HPD proxy for P1-P7 and XLNet for P8.
    Xtr_emb = encode_split_for_prompt(tr_df, prompt_id)
    Xva_emb = encode_split_for_prompt(va_df, prompt_id)
    Xte_emb = encode_split_for_prompt(te_df, prompt_id)

    # 2) Scale discourse features using train only.
    scaler = StandardScaler()
    Xtr_feat = scaler.fit_transform(tr_feat.values)
    Xva_feat = scaler.transform(va_feat.values)
    Xte_feat = scaler.transform(te_feat.values)

    # 3) Concatenate embedding + discourse features.
    Xtr = np.hstack([Xtr_emb, Xtr_feat])
    Xva = np.hstack([Xva_emb, Xva_feat])
    Xte = np.hstack([Xte_emb, Xte_feat])

    # 4) Normalize labels to [0, 1] for stable regression head training.
    ytr_raw = tr_df[SCORE_COL].values.astype(float)
    yva_raw = va_df[SCORE_COL].values.astype(float)
    yte_raw = te_df[SCORE_COL].values.astype(float)

    ytr = (ytr_raw - min_score) / max(1, score_range)
    yva = (yva_raw - min_score) / max(1, score_range)

    # 5) Train linear head.
    head, hist = train_linear_head(Xtr, ytr, Xva, yva)
    hist[ESSAY_SET_COL] = prompt_id
    all_histories.append(hist)

    # 6) Predict and denormalize.
    va_pred_norm = predict_head(head, Xva)
    te_pred_norm = predict_head(head, Xte)

    va_pred_raw = va_pred_norm * max(1, score_range) + min_score
    te_pred_raw = te_pred_norm * max(1, score_range) + min_score

    # 7) Optimize thresholds on validation, then apply to test.
    thresholds = optimize_thresholds(yva_raw.astype(int), va_pred_raw, min_score, max_score)

    val_eval = evaluate_predictions(yva_raw.astype(int), va_pred_raw, min_score, max_score, thresholds)
    test_eval = evaluate_predictions(yte_raw.astype(int), te_pred_raw, min_score, max_score, thresholds)

    result = {
        ESSAY_SET_COL: int(prompt_id),
        'method': 'XHPDF-style Reproduced',
        'embedding_model': XLNET_MODEL_NAME if int(prompt_id) == 8 else HPD_MODEL_NAME,
        'feature_count': Xtr_feat.shape[1],
        'embedding_dim': Xtr_emb.shape[1],
        'concat_dim': Xtr.shape[1],
        'min_score': min_score,
        'max_score': max_score,
        'val_qwk_round': val_eval['qwk_round'],
        'val_qwk': val_eval.get('qwk_opt', np.nan),
        'test_qwk_round': test_eval['qwk_round'],
        'test_qwk': test_eval.get('qwk_opt', np.nan),
        'thresholds': thresholds.tolist(),
    }
    all_results.append(result)
    print(result)

    # Save test predictions for this prompt.
    pred_df = te_df.copy()
    pred_df['raw_prediction'] = te_pred_raw
    pred_df['pred_round'] = clip_and_round(te_pred_raw, min_score, max_score)
    pred_df['pred_opt'] = apply_thresholds(te_pred_raw, thresholds, min_score, max_score)
    pred_df['method'] = 'XHPDF-style Reproduced'
    all_predictions.append(pred_df)

results_df = pd.DataFrame(all_results)
predictions_df = pd.concat(all_predictions, ignore_index=True)
history_df = pd.concat(all_histories, ignore_index=True)

display(results_df)



PROMPT 1
Score range: 2 to 12
Rows: 1248 180 355


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding sentence-transformers/all-MiniLM-L6-v2:   0%|          | 0/19 [00:00<?, ?it/s]

Epoch 001 | train_loss=0.21560 | val_loss=0.12551
Epoch 010 | train_loss=0.00834 | val_loss=0.00632
Epoch 020 | train_loss=0.00569 | val_loss=0.00491
Epoch 030 | train_loss=0.00470 | val_loss=0.00494
Early stopping at epoch 36
{'essay_set': 8, 'method': 'XHPDF-style Reproduced', 'embedding_model': 'xlnet-base-cased', 'feature_count': 31, 'embedding_dim': 384, 'concat_dim': 415, 'min_score': 10, 'max_score': 60, 'val_qwk_round': np.float64(0.7265418952653625), 'val_qwk': np.float64(0.7620926507452421), 'test_qwk_round': np.float64(0.6754182421148147), 'test_qwk': np.float64(0.6807877738780452), 'thresholds': [10.571934876715567, 11.476280526097355, 12.589527113696718, 13.484364690477186, 14.462684059724232, 15.475162264095735, 16.610007352025022, 17.62927089691658, 18.542873720972302, 19.5129624040462, 20.523816148455552, 21.474317737528928, 22.509753671880226, 23.48987291495176, 24.52725699303835, 25.46312549757759, 26.705508090728177, 27.71935921596535, 28.411253051076613, 29.71788706

,essay_set,method,embedding_model,feature_count,embedding_dim,concat_dim,min_score,max_score,val_qwk_round,val_qwk,test_qwk_round,test_qwk,thresholds
0,1,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,2,12,0.836679,0.862533,0.791904,0.782242,"[2.4487033931268862, 3.4745200334075834, 4.749..."
1,2,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,1,6,0.703880,0.735234,0.694846,0.697252,"[1.5184495782000003, 2.499620697, 3.4759804169..."
2,3,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,0,3,0.656777,0.718730,0.679821,0.727813,"[0.5064814814814814, 1.6284722222222223, 2.244..."
3,4,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,0,3,0.772045,0.792782,0.779261,0.802157,"[0.5152027606310013, 1.5046489197530886, 2.312..."
4,5,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,0,4,0.835008,0.837924,0.785705,0.795782,"[0.4906158447265624, 1.4727447509765628, 2.581..."
5,6,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,0,4,0.817961,0.847323,0.787527,0.786961,"[0.5472220420837408, 1.5668904304504396, 2.278..."
6,7,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31,384,415,2,24,0.784218,0.790165,0.779688,0.777510,"[2.503126333070462, 3.5043768662986468, 4.5056..."
7,8,XHPDF-style Reproduced,xlnet-base-cased,31,384,415,10,60,0.726542,0.762093,0.675418,0.680788,"[10.571934876715567, 11.476280526097355, 12.58..."


In [10]:
# ============================================================
# 9. AGGREGATE RESULTS AND SAVE
# ============================================================
avg_row = {
    ESSAY_SET_COL: 'Avg',
    'method': 'XHPDF-style Reproduced',
    'embedding_model': 'P1-P7 HPD proxy; P8 XLNet',
    'feature_count': results_df['feature_count'].mean(),
    'embedding_dim': np.nan,
    'concat_dim': np.nan,
    'min_score': np.nan,
    'max_score': np.nan,
    'val_qwk_round': results_df['val_qwk_round'].mean(),
    'val_qwk': results_df['val_qwk'].mean(),
    'test_qwk_round': results_df['test_qwk_round'].mean(),
    'test_qwk': results_df['test_qwk'].mean(),
    'thresholds': '',
}

results_with_avg = pd.concat([results_df, pd.DataFrame([avg_row])], ignore_index=True)

results_path = OUTPUT_DIR / 'xhpdf_style_reproduced_results.csv'
pred_path = OUTPUT_DIR / 'xhpdf_style_reproduced_predictions.csv'
hist_path = OUTPUT_DIR / 'xhpdf_style_reproduced_training_history.csv'
feat_path = OUTPUT_DIR / 'xhpdf_style_extracted_discourse_features.csv'

results_with_avg.to_csv(results_path, index=False)
predictions_df.to_csv(pred_path, index=False)
history_df.to_csv(hist_path, index=False)

# Save features with split labels for inspection.
features_all = pd.concat([
    pd.concat([train_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), train_feat_df.reset_index(drop=True)], axis=1).assign(split='train'),
    pd.concat([val_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), val_feat_df.reset_index(drop=True)], axis=1).assign(split='val'),
    pd.concat([test_df[[ESSAY_SET_COL, SCORE_COL]].reset_index(drop=True), test_feat_df.reset_index(drop=True)], axis=1).assign(split='test'),
], ignore_index=True)
features_all.to_csv(feat_path, index=False)

print('Saved:')
print(results_path)
print(pred_path)
print(hist_path)
print(feat_path)

display(results_with_avg)

# Handy row for your paper-style table.
table_row_round = ['XHPDF-style Reproduced (round)'] + [
    float(results_df.loc[results_df[ESSAY_SET_COL] == p, 'test_qwk_round'].iloc[0])
    for p in sorted(train_df[ESSAY_SET_COL].unique())
] + [float(results_df['test_qwk_round'].mean())]

table_row_opt = ['XHPDF-style Reproduced (opt)'] + [
    float(results_df.loc[results_df[ESSAY_SET_COL] == p, 'test_qwk'].iloc[0])
    for p in sorted(train_df[ESSAY_SET_COL].unique())
] + [float(results_df['test_qwk'].mean())]

print('Table row using round:')
print(table_row_round)
print('Table row using opt thresholds:')
print(table_row_opt)


Saved:
/content/benchmark_results/ASAP/xhpdf_style_reproduced_results.csv
/content/benchmark_results/ASAP/xhpdf_style_reproduced_predictions.csv
/content/benchmark_results/ASAP/xhpdf_style_reproduced_training_history.csv
/content/benchmark_results/ASAP/xhpdf_style_extracted_discourse_features.csv


,essay_set,method,embedding_model,feature_count,embedding_dim,concat_dim,min_score,max_score,val_qwk_round,val_qwk,test_qwk_round,test_qwk,thresholds
0,1,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,2.0,12.0,0.836679,0.862533,0.791904,0.782242,"[2.4487033931268862, 3.4745200334075834, 4.749..."
1,2,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,1.0,6.0,0.703880,0.735234,0.694846,0.697252,"[1.5184495782000003, 2.499620697, 3.4759804169..."
2,3,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,0.0,3.0,0.656777,0.718730,0.679821,0.727813,"[0.5064814814814814, 1.6284722222222223, 2.244..."
3,4,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,0.0,3.0,0.772045,0.792782,0.779261,0.802157,"[0.5152027606310013, 1.5046489197530886, 2.312..."
4,5,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,0.0,4.0,0.835008,0.837924,0.785705,0.795782,"[0.4906158447265624, 1.4727447509765628, 2.581..."
5,6,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,0.0,4.0,0.817961,0.847323,0.787527,0.786961,"[0.5472220420837408, 1.5668904304504396, 2.278..."
6,7,XHPDF-style Reproduced,sentence-transformers/all-MiniLM-L6-v2,31.0,384.0,415.0,2.0,24.0,0.784218,0.790165,0.779688,0.777510,"[2.503126333070462, 3.5043768662986468, 4.5056..."
7,8,XHPDF-style Reproduced,xlnet-base-cased,31.0,384.0,415.0,10.0,60.0,0.726542,0.762093,0.675418,0.680788,"[10.571934876715567, 11.476280526097355, 12.58..."
8,Avg,XHPDF-style Reproduced,P1-P7 HPD proxy; P8 XLNet,31.0,NaN,NaN,NaN,NaN,0.766639,0.793348,0.746771,0.756313,


Table row using round:
['XHPDF-style Reproduced (round)', 0.7919042776532368, 0.6948457133088297, 0.6798214601378822, 0.7792612070882357, 0.7857052891437784, 0.7875270526920188, 0.7796877171608829, 0.6754182421148147, 0.7467713699124598]
Table row using opt thresholds:
['XHPDF-style Reproduced (opt)', 0.7822419874252415, 0.6972523744911805, 0.7278132278037605, 0.8021574437130742, 0.7957821155591751, 0.7869613030397344, 0.7775097665005828, 0.6807877738780452, 0.7563132490513493]


## Reporting note

Use:

- `test_qwk_round` for a cleaner comparison against simple regression baselines.
- `test_qwk` only if you clearly say it uses **validation threshold optimization**.

Recommended method name in your table:

- **XHPDF-style Reproduced** if using the default HPD proxy.
- **XHPDF Reproduced** only if you replace `HPD_MODEL_NAME` with the exact HPD model/checkpoint and match the paper settings closely.
